# ASOS (종관기상관측) 데이터 EDA
## 기상청 API 허브 — 시간자료(기간 조회) 기반 탐색적 데이터 분석

---

### API 엔드포인트
```
https://apihub.kma.go.kr/api/typ01/url/kma_sfctm3.php
  ?tm1=YYYYMMDDHHMM  시작시각 (KST)
  &tm2=YYYYMMDDHHMM  종료시각 (KST, 최대 31일)
  &stn=108           지점번호 (0=전체)
  &help=1            헤더 포함
  &authKey=...       인증키
```

### 주요 출력 변수

| 변수 | 의미 | 단위 |
|------|------|------|
| TM | 관측시각 (KST) | YYYYMMDDHHmm |
| STN | 지점번호 | - |
| TA | 기온 | °C |
| HM | 상대습도 | % |
| RN | 강수량 | mm |
| WS | 풍속 | m/s |
| WD | 풍향 | 36방위 |
| PA | 현지기압 | hPa |
| PS | 해면기압 | hPa |
| TD | 이슬점온도 | °C |
| PV | 수증기압 | hPa |
| SS | 일조 | hr |
| SI | 일사 | MJ/m² |
| TS | 지면온도 | °C |
| SD_TOT | 적설 | cm |

---
## Step 1. 패키지 임포트 및 환경 설정

In [ ]:
# !pip install -q requests pandas numpy matplotlib seaborn tqdm

import pathlib, matplotlib, matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import urllib.request

font_path = pathlib.Path(matplotlib.get_data_path()) / 'fonts' / 'ttf' / 'NanumGothic.ttf'
if not font_path.exists():
    try:
        urllib.request.urlretrieve(
            'https://raw.githubusercontent.com/google/fonts/main/ofl/nanumgothic/NanumGothic-Regular.ttf',
            font_path
        )
    except:
        pass

if font_path.exists():
    fm.fontManager.addfont(str(font_path))
    plt.rcParams['font.family'] = fm.FontProperties(fname=str(font_path)).get_name()
plt.rcParams['axes.unicode_minus'] = False
print(f'폰트: {plt.rcParams["font.family"]}')

In [ ]:
import requests
import io
import os
import time
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import warnings
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

import pathlib, matplotlib, matplotlib.font_manager as fm
font_path = pathlib.Path(matplotlib.get_data_path()) / 'fonts' / 'ttf' / 'NanumGothic.ttf'
if font_path.exists():
    fm.fontManager.addfont(str(font_path))
    plt.rcParams['font.family'] = fm.FontProperties(fname=str(font_path)).get_name()
plt.rcParams['axes.unicode_minus'] = False

print(f'pandas : {pd.__version__}')
print(f'numpy  : {np.__version__}')

---
## Step 2. 2020년 1년치 시간자료 다운로드

- API 최대 조회 기간이 **31일**이므로 월별로 나눠 12회 요청합니다.
- 응답은 공백 구분 텍스트이며 `#` 으로 시작하는 줄은 주석(헤더)입니다.
- 다운로드 결과는 `./asos_data/` 에 월별 CSV로 저장합니다.

In [ ]:
# ── API 설정 ─────────────────────────────────────────────────────────
import getpass

AUTH_KEY = getpass.getpass('기상청 API 인증키를 입력하세요: ')  # 입력값은 화면에 표시되지 않음

BASE_URL  = 'https://apihub.kma.go.kr/api/typ01/url/kma_sfctm3.php'
STN       = 108        # 지점번호 (108=서울, 0=전체 지점)
YEAR      = 2020       # 다운로드 연도
SAVE_DIR  = './asos_data'

os.makedirs(SAVE_DIR, exist_ok=True)

# API 응답 컬럼 순서 (help=1 헤더 기준)
ASOS_COLS = [
    'TM','STN',
    'WD','WS','GST_WD','GST_WS','GST_TM',
    'PA','PS','PT','PR',
    'TA','TD','HM','PV',
    'RN','RN_DAY','RN_INT',
    'SD_HR3','SD_DAY','SD_TOT',
    'WC','WP','WW',
    'CA_TOT','CA_MID','CH_MIN','CT','CT_TOP','CT_MID','CT_LOW',
    'VS','SS','SI',
    'ST_GD','TS','TE_005','TE_01','TE_02','TE_03',
    'ST_SEA','WH','BF','IR','IX','RN_JUN'
]

print(f'인증키 입력 완료')
print(f'대상: {YEAR}년 전체 / 지점: {STN} / 저장: {SAVE_DIR}')
print(f'컬럼 수: {len(ASOS_COLS)}개')

In [ ]:
# ── 응답 텍스트 파싱 함수 ─────────────────────────────────────────────
def parse_asos_response(text):
    """
    기상청 API 응답 텍스트 → DataFrame
    - '#' 로 시작하는 줄은 주석/헤더로 건너뜀
    - 데이터는 공백 구분, 결측값은 '-9', '-99', '-999' 등
    """
    lines = []
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        lines.append(line)

    if not lines:
        return pd.DataFrame(columns=ASOS_COLS)

    records = []
    for line in lines:
        parts = line.split()
        # 컬럼 수에 맞게 패딩 또는 트림
        if len(parts) < len(ASOS_COLS):
            parts += [np.nan] * (len(ASOS_COLS) - len(parts))
        records.append(parts[:len(ASOS_COLS)])

    df = pd.DataFrame(records, columns=ASOS_COLS)
    return df


def fetch_asos(tm1, tm2, stn=STN, auth_key=AUTH_KEY):
    """
    tm1, tm2: datetime 객체 또는 'YYYYMMDDHHMM' 문자열
    """
    if isinstance(tm1, datetime):
        tm1 = tm1.strftime('%Y%m%d%H%M')
    if isinstance(tm2, datetime):
        tm2 = tm2.strftime('%Y%m%d%H%M')

    params = {
        'tm1'    : tm1,
        'tm2'    : tm2,
        'stn'    : stn,
        'help'   : 1,
        'authKey': auth_key,
    }
    resp = requests.get(BASE_URL, params=params, timeout=60)
    resp.raise_for_status()
    return parse_asos_response(resp.text)


print('파싱 함수 정의 완료')

In [ ]:
# ── 월별 다운로드 루프 (2020년 1~12월) ──────────────────────────────
monthly_dfs = []

for month in tqdm(range(1, 13), desc='월별 다운로드'):
    tm1 = datetime(YEAR, month, 1, 0, 0)
    # 월의 마지막 날 23시
    if month == 12:
        tm2 = datetime(YEAR, 12, 31, 23, 0)
    else:
        tm2 = datetime(YEAR, month + 1, 1, 0, 0) - timedelta(hours=1)

    save_path = os.path.join(SAVE_DIR, f'asos_{YEAR}{month:02d}.csv')

    # 이미 저장된 파일이 있으면 재사용
    if os.path.exists(save_path):
        df_month = pd.read_csv(save_path, dtype=str)
        print(f'  {YEAR}-{month:02d}  캐시 로드  ({len(df_month)}행)')
    else:
        df_month = fetch_asos(tm1, tm2)
        df_month.to_csv(save_path, index=False)
        print(f'  {YEAR}-{month:02d}  다운로드 완료  ({len(df_month)}행)')
        time.sleep(0.5)   # API 호출 간격

    monthly_dfs.append(df_month)

df_raw = pd.concat(monthly_dfs, ignore_index=True)
print(f'\n전체 로드 완료: {df_raw.shape}')

In [ ]:
# ── 월별 파일 → 연간 단일 파일로 합치기 ─────────────────────────────
merged_path = os.path.join(SAVE_DIR, f'asos_{YEAR}_merged.csv')

df_raw.to_csv(merged_path, index=False)

size_mb = os.path.getsize(merged_path) / 1e6
print(f'저장 완료: {merged_path}')
print(f'파일 크기: {size_mb:.1f} MB')
print(f'총 행 수 : {len(df_raw):,}행  /  컬럼 수: {df_raw.shape[1]}개')

---
## Step 3. 데이터 전처리

In [ ]:
# ── 타입 변환 및 결측값 처리 ──────────────────────────────────────────
df = df_raw.copy()

# datetime 파싱 (TM: YYYYMMDDHHmm)
df['datetime'] = pd.to_datetime(df['TM'], format='%Y%m%d%H%M', errors='coerce')
df = df.sort_values('datetime').reset_index(drop=True)

# 수치형 컬럼 변환 + 결측 기호(-9, -99, -999, -9999 등) → NaN
NUM_COLS = [
    'WD','WS','GST_WS','PA','PS','PR',
    'TA','TD','HM','PV',
    'RN','RN_DAY','RN_INT',
    'SD_HR3','SD_DAY','SD_TOT',
    'CA_TOT','CA_MID','CH_MIN',
    'VS','SS','SI',
    'TS','TE_005','TE_01','TE_02','TE_03',
    'WH'
]

for col in NUM_COLS:
    if col not in df.columns:
        continue
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df.loc[df[col] <= -9, col] = np.nan

# 시간 파생 변수
df['year']   = df['datetime'].dt.year
df['month']  = df['datetime'].dt.month
df['day']    = df['datetime'].dt.day
df['hour']   = df['datetime'].dt.hour
df['doy']    = df['datetime'].dt.dayofyear
df['season'] = df['month'].map({
    12:'겨울',1:'겨울',2:'겨울',
    3:'봄',  4:'봄',  5:'봄',
    6:'여름',7:'여름',8:'여름',
    9:'가을',10:'가을',11:'가을'
})

# ── 변수 한글명 매핑 (전역 사용) ─────────────────────────────────────
COL_KR = {
    'TA':'기온(°C)','HM':'습도(%)','RN':'강수량(mm)','WS':'풍속(m/s)',
    'WD':'풍향(36방위)','GST_WS':'돌풍속(m/s)',
    'PA':'현지기압(hPa)','PS':'해면기압(hPa)','PR':'기압변화량(hPa)',
    'TD':'이슬점(°C)','PV':'수증기압(hPa)',
    'RN_DAY':'일강수량(mm)','RN_INT':'강수강도(mm/h)',
    'SD_HR3':'3시간신적설(cm)','SD_DAY':'일신적설(cm)','SD_TOT':'적설(cm)',
    'CA_TOT':'전운량(1/10)','CA_MID':'중하층운량(1/10)','CH_MIN':'최저운고(100m)',
    'VS':'시정(10m)','SS':'일조(hr)','SI':'일사(MJ/m²)',
    'TS':'지면온도(°C)',
    'TE_005':'5cm지중온도(°C)','TE_01':'10cm지중온도(°C)',
    'TE_02':'20cm지중온도(°C)','TE_03':'30cm지중온도(°C)',
    'WH':'파고(m)',
}

print(f'전처리 완료: {df.shape}')
print(f'기간: {df["datetime"].min()} ~ {df["datetime"].max()}')
print(f'지점: {df["STN"].unique()}')
df[['datetime','STN','TA','HM','RN','WS','PA','PS','TD','SS']].head(10)

---
## Step 4. 결측치 분석

In [ ]:
# ── 결측치 요약 ───────────────────────────────────────────────────────
plot_cols = [c for c in NUM_COLS if c in df.columns]

miss_df = pd.DataFrame({
    '결측 수'  : df[plot_cols].isna().sum(),
    '결측률(%)': (df[plot_cols].isna().mean() * 100).round(2),
}).sort_values('결측률(%)', ascending=False)

# 변수명(한글) 컬럼 추가
miss_df.insert(0, '변수명', [COL_KR.get(c, '-') for c in miss_df.index])
print(miss_df.to_string())

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#E63946' if v > 20 else '#FF9800' if v > 5 else '#2196F3'
          for v in miss_df['결측률(%)']]

# y축 레이블: 변수코드 + 한글명
ylabels = [f"{code}  {COL_KR.get(code, '')}" for code in miss_df.index]

bars = ax.barh(ylabels, miss_df['결측률(%)'], color=colors, edgecolor='white')
ax.bar_label(bars, fmt='%.1f%%', fontsize=8, padding=3)
ax.set_xlabel('결측률 (%)')
ax.set_title(f'ASOS 변수별 결측률 ({YEAR}년, 지점 {STN})', fontsize=13)
ax.axvline(5,  color='orange', linestyle='--', lw=1.2, label='5%')
ax.axvline(20, color='red',    linestyle='--', lw=1.2, label='20%')
ax.legend()
plt.tight_layout()
plt.show()

---
## Step 5. 기본 통계 및 분포

In [ ]:
# ── 기본 통계 ─────────────────────────────────────────────────────────
key_cols = [c for c in ['TA','HM','RN','WS','PA','PS','TD','PV','SS','SI','TS']
            if c in df.columns]

COL_KR = {
    'TA':'기온(°C)','HM':'습도(%)','RN':'강수량(mm)','WS':'풍속(m/s)',
    'PA':'현지기압(hPa)','PS':'해면기압(hPa)','TD':'이슬점(°C)',
    'PV':'수증기압(hPa)','SS':'일조(hr)','SI':'일사(MJ/m²)',
    'TS':'지면온도(°C)','SD_TOT':'적설(cm)'
}

desc = df[key_cols].describe().T
desc.index = [COL_KR.get(c, c) for c in desc.index]
print(desc.round(2).to_string())

In [ ]:
# ── 주요 변수 분포 히스토그램 ─────────────────────────────────────────
PLOT_VARS = [c for c in ['TA','HM','RN','WS','PS','TD'] if c in df.columns]
COLORS    = ['#E63946','#4CAF50','#2196F3','#FF9800','#9C27B0','#00BCD4']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes_flat = axes.ravel()

for i, col in enumerate(PLOT_VARS):
    ax   = axes_flat[i]
    data = df[col].dropna()
    label = COL_KR.get(col, col)

    if col == 'RN':
        data_nz = data[data > 0]
        ax.hist(data_nz, bins=60, color=COLORS[i], edgecolor='white',
                density=True, alpha=0.85)
        ax.set_title(f'{label}\n(강수 발생 시간 n={len(data_nz):,})', fontsize=11)
    else:
        ax.hist(data, bins=60, color=COLORS[i], edgecolor='white',
                density=True, alpha=0.85)
        ax.set_title(f'{label}  (n={len(data):,})', fontsize=11)

    ax.axvline(data.mean(),   color='red',  linestyle='--', lw=1.5,
               label=f'평균={data.mean():.1f}')
    ax.axvline(data.median(), color='lime', linestyle=':',  lw=1.5,
               label=f'중앙={data.median():.1f}')
    ax.set_xlabel(label, fontsize=9)
    ax.set_ylabel('밀도', fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle(f'ASOS 주요 변수 분포 ({YEAR}년, 지점 {STN})', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## Step 6. 시계열 시각화

In [ ]:
# ── 일평균 집계 ───────────────────────────────────────────────────────
agg_dict = {}
if 'TA' in df.columns:
    agg_dict.update(temp_mean=('TA','mean'), temp_max=('TA','max'), temp_min=('TA','min'))
if 'RN' in df.columns:
    agg_dict['precip_sum'] = ('RN','sum')
if 'HM' in df.columns:
    agg_dict['hm_mean'] = ('HM','mean')
if 'WS' in df.columns:
    agg_dict['ws_mean'] = ('WS','mean')
if 'PS' in df.columns:
    agg_dict['ps_mean'] = ('PS','mean')

df_daily = df.groupby(df['datetime'].dt.date).agg(**agg_dict).reset_index()
df_daily['datetime'] = pd.to_datetime(df_daily['datetime'])

# ── 시계열 플롯 ───────────────────────────────────────────────────────
n_panels = sum([
    'temp_mean' in df_daily.columns,
    'precip_sum' in df_daily.columns,
    'hm_mean' in df_daily.columns,
    'ps_mean' in df_daily.columns,
])

fig, axes = plt.subplots(n_panels, 1, figsize=(16, 3.5 * n_panels), sharex=True)
if n_panels == 1:
    axes = [axes]
ax_i = 0

if 'temp_mean' in df_daily.columns:
    axes[ax_i].plot(df_daily['datetime'], df_daily['temp_mean'],
                    color='tomato', lw=1.2, label='일평균')
    axes[ax_i].fill_between(df_daily['datetime'],
                             df_daily['temp_min'], df_daily['temp_max'],
                             alpha=0.2, color='tomato', label='최저~최고')
    axes[ax_i].set_ylabel('기온 (°C)')
    axes[ax_i].set_title('일평균 기온', fontsize=11)
    axes[ax_i].legend(fontsize=9)
    axes[ax_i].grid(alpha=0.3)
    ax_i += 1

if 'precip_sum' in df_daily.columns:
    axes[ax_i].bar(df_daily['datetime'], df_daily['precip_sum'],
                   color='steelblue', width=1, alpha=0.8)
    axes[ax_i].set_ylabel('강수량 (mm)')
    axes[ax_i].set_title('일강수량', fontsize=11)
    axes[ax_i].grid(alpha=0.3)
    ax_i += 1

if 'hm_mean' in df_daily.columns:
    axes[ax_i].plot(df_daily['datetime'], df_daily['hm_mean'],
                    color='seagreen', lw=1.2)
    axes[ax_i].set_ylabel('습도 (%)')
    axes[ax_i].set_title('일평균 상대습도', fontsize=11)
    axes[ax_i].grid(alpha=0.3)
    ax_i += 1

if 'ps_mean' in df_daily.columns:
    axes[ax_i].plot(df_daily['datetime'], df_daily['ps_mean'],
                    color='purple', lw=1.2)
    axes[ax_i].set_ylabel('해면기압 (hPa)')
    axes[ax_i].set_title('일평균 해면기압', fontsize=11)
    axes[ax_i].set_xlabel('날짜')
    axes[ax_i].grid(alpha=0.3)

plt.suptitle(f'ASOS 주요 변수 시계열 — {YEAR}년, 지점 {STN}', fontsize=14)
plt.tight_layout()
plt.show()

---
## Step 7. 계절·월별 패턴 분석

In [ ]:
# ── 월별·계절별 패턴 ──────────────────────────────────────────────────
month_names = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']
season_order  = ['봄','여름','가을','겨울']
season_colors = {'봄':'#4CAF50','여름':'#E63946','가을':'#FF9800','겨울':'#2196F3'}

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 월별 기온 박스플롯
if 'TA' in df.columns:
    month_groups = [df[df['month'] == m]['TA'].dropna() for m in range(1, 13)]
    bp = axes[0,0].boxplot(month_groups, patch_artist=True,
                           medianprops={'color':'red','lw':2})
    for patch, c in zip(bp['boxes'], plt.cm.RdYlBu_r(np.linspace(0.1, 0.9, 12))):
        patch.set_facecolor(c)
    axes[0,0].set_xticklabels(month_names, rotation=45, fontsize=9)
    axes[0,0].set_ylabel('기온 (°C)')
    axes[0,0].set_title('월별 기온 분포', fontsize=12)
    axes[0,0].grid(axis='y', alpha=0.3)

# 월별 총 강수량
if 'RN' in df.columns:
    monthly_rn = df.groupby('month')['RN'].sum()
    axes[0,1].bar(monthly_rn.index, monthly_rn.values,
                  color='steelblue', edgecolor='white')
    axes[0,1].set_xticks(range(1,13))
    axes[0,1].set_xticklabels(month_names, rotation=45, fontsize=9)
    axes[0,1].set_ylabel('총 강수량 (mm)')
    axes[0,1].set_title('월별 총 강수량', fontsize=12)
    axes[0,1].bar_label(axes[0,1].containers[0], fmt='%.0f', fontsize=8)
    axes[0,1].grid(axis='y', alpha=0.3)

# 시간대별 평균 기온
if 'TA' in df.columns:
    hourly_ta = df.groupby('hour')['TA'].mean()
    axes[1,0].plot(hourly_ta.index, hourly_ta.values,
                   'o-', color='tomato', lw=2, ms=6)
    axes[1,0].set_xlabel('시각 (KST)')
    axes[1,0].set_ylabel('평균 기온 (°C)')
    axes[1,0].set_title('시간대별 평균 기온', fontsize=12)
    axes[1,0].set_xticks(range(0, 24, 3))
    axes[1,0].grid(alpha=0.3)

# 계절별 기온 바이올린
if 'TA' in df.columns:
    sns.violinplot(
        data=df[df['season'].notna()],
        x='season', y='TA',
        order=season_order,
        palette=season_colors,
        ax=axes[1,1], inner='quartile'
    )
    axes[1,1].set_xlabel('')
    axes[1,1].set_ylabel('기온 (°C)')
    axes[1,1].set_title('계절별 기온 분포', fontsize=12)
    axes[1,1].grid(axis='y', alpha=0.3)

plt.suptitle(f'ASOS 계절·월별 패턴 — {YEAR}년, 지점 {STN}', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── 월 × 시간대 히트맵 (기온) ─────────────────────────────────────────
if 'TA' in df.columns:
    pivot = df.pivot_table(index='hour', columns='month', values='TA', aggfunc='mean')
    pivot.columns = month_names

    fig, ax = plt.subplots(figsize=(14, 6))
    sns.heatmap(pivot, ax=ax, cmap='RdYlBu_r',
                annot=True, fmt='.1f', linewidths=0.3,
                annot_kws={'size': 8},
                cbar_kws={'label': '평균 기온 (°C)'})
    ax.set_xlabel('월')
    ax.set_ylabel('시각 (KST)')
    ax.set_title(f'월 × 시간대별 평균 기온 히트맵 — {YEAR}년, 지점 {STN}', fontsize=13)
    plt.tight_layout()
    plt.show()

---
## Step 8. 상관 분석

In [ ]:
# ── 변수 간 상관계수 ──────────────────────────────────────────────────
corr_vars = [c for c in ['TA','HM','RN','WS','PS','TD','PV','SS','SI','TS']
             if c in df.columns]

corr_mat = df[corr_vars].corr()
kr_labels = [COL_KR.get(c, c) for c in corr_mat.columns]
corr_mat.columns = kr_labels
corr_mat.index   = kr_labels

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(
    corr_mat, ax=axes[0],
    annot=True, fmt='.2f', cmap='RdBu_r',
    vmin=-1, vmax=1, linewidths=0.4,
    annot_kws={'size': 9}
)
axes[0].set_title('변수 간 피어슨 상관계수', fontsize=12)

# 기온 vs 이슬점 산점도 (색상=습도)
if 'TA' in df.columns and 'TD' in df.columns and 'HM' in df.columns:
    sample = df[['TA','TD','HM']].dropna().sample(min(5000, len(df)))
    sc = axes[1].scatter(sample['TA'], sample['TD'],
                         c=sample['HM'], cmap='YlGnBu',
                         s=3, alpha=0.5, vmin=20, vmax=100)
    plt.colorbar(sc, ax=axes[1], label='습도 (%)')
    r = df[['TA','TD']].dropna().corr().iloc[0,1]
    axes[1].set_xlabel('기온 (°C)', fontsize=11)
    axes[1].set_ylabel('이슬점온도 (°C)', fontsize=11)
    axes[1].set_title(f'기온 vs 이슬점  (r={r:.3f})\n색상=습도', fontsize=12)
    axes[1].grid(alpha=0.3)

plt.suptitle(f'ASOS 변수 간 상관 분석 — {YEAR}년, 지점 {STN}', fontsize=14)
plt.tight_layout()
plt.show()

---
## Step 9. 이상치 탐색

In [ ]:
# ── IQR 기반 이상치 요약 ──────────────────────────────────────────────
out_vars = [c for c in ['TA','HM','RN','WS','PS','TS'] if c in df.columns]

print(f"{'변수':<18} {'Q1':>7} {'Q3':>7} {'하한':>9} {'상한':>9} "
      f"{'이상치수':>9} {'이상치율%':>10}")
print('-' * 74)

for col in out_vars:
    s   = df[col].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
    n_out = ((s < lo) | (s > hi)).sum()
    print(f"{COL_KR.get(col,col):<18} {q1:>7.2f} {q3:>7.2f} "
          f"{lo:>9.2f} {hi:>9.2f} {n_out:>9,} {n_out/len(s)*100:>9.2f}%")

# 박스플롯
fig, axes = plt.subplots(1, len(out_vars), figsize=(3.5 * len(out_vars), 5))
if len(out_vars) == 1:
    axes = [axes]
for ax, col in zip(axes, out_vars):
    ax.boxplot(df[col].dropna(), patch_artist=True,
               boxprops={'facecolor':'#2196F3','alpha':0.7},
               medianprops={'color':'red','lw':2},
               flierprops={'marker':'o','markersize':2,'alpha':0.3})
    ax.set_title(COL_KR.get(col, col), fontsize=11)
    ax.set_xticks([])
    ax.grid(axis='y', alpha=0.3)

plt.suptitle(f'이상치 박스플롯 (IQR 기준) — {YEAR}년, 지점 {STN}', fontsize=13)
plt.tight_layout()
plt.show()

---
## Step 10. 요약 및 DL 학습 노트

In [ ]:
# ── DL 학습용 정규화 통계 ─────────────────────────────────────────────
norm_vars = [c for c in ['TA','HM','RN','WS','PS','TD','PV','SS','SI','TS']
             if c in df.columns]

print('=' * 72)
print(f'DL 학습 정규화 참고값  ({YEAR}년, 지점 {STN})')
print('=' * 72)
print(f"{'변수':<18} {'평균':>8} {'표준편차':>9} {'최솟값':>8} "
      f"{'1%':>8} {'99%':>8} {'최댓값':>8}")
print('-' * 72)

for col in norm_vars:
    s = df[col].dropna()
    print(f"{COL_KR.get(col,col):<18} "
          f"{s.mean():>8.2f} {s.std():>9.2f} {s.min():>8.2f} "
          f"{s.quantile(0.01):>8.2f} {s.quantile(0.99):>8.2f} {s.max():>8.2f}")

print()
print('전처리 권장사항:')
notes = [
    'TA / TD / TS  : Z-score 정규화 권장',
    'HM            : 0~100% → /100 단순 정규화 가능',
    'RN            : 0이 압도적 → log1p 변환 후 정규화',
    'WS            : 0 이상 비대칭 → sqrt 또는 log1p 변환 고려',
    'PS / PA       : 변동폭 작음 → Z-score 또는 편차 사용',
    'SS / SI       : 야간 0 포함 → log1p 변환 고려',
    '결측치        : 시계열 선형 보간(interpolate) 또는 마스킹 처리',
]
for note in notes:
    print(f'  - {note}')